# FAPI radiomics – four experiments (all foci)
Runs the complete workflow with the scripts in this folder. Set `DATA_DIR` to the folder containing `CT.csv`, `PET_64.csv`, `PET_128.csv` (on Kaggle: `/kaggle/input/<dataset>`).

In [ ]:
import os, subprocess, sys
DATA_DIR = os.environ.get('FAPI_DATA', r'D:/FAPI-Seg/Radiomics/Modeling/Final_Excels/Target_Assigned')
OUT = '..'
!{sys.executable} -m pip install -q -r requirements.txt

In [ ]:
def run(*args):
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-2000:])
    assert r.returncode == 0
run('build_datasets.py', '--src', DATA_DIR, '--out', '../01_data')

## Four experiments + label-permutation leakage checks (run in parallel)

In [ ]:
procs = []
for e in ['exp1_pet64', 'exp2_pet128', 'exp3_pet64_ct', 'exp4_pet128_ct']:
    for extra in ([], ['--permute', '1']):
        procs.append(subprocess.Popen([sys.executable, 'run_experiment.py', '--exp', e, *extra]))
assert all(p.wait() == 0 for p in procs)

## Reference models, summaries, figures, final-model SHAP, tables

In [ ]:
for s in ['reference_models.py', 'summarize.py', 'figures.py', 'final_model_shap.py', 'make_tables.py']:
    run(s)

In [ ]:
import pandas as pd
pd.read_csv('../tables/T4_consolidated_best_per_experiment.csv')